In [ ]:
import pandas as pd
import requests

##Rankings

**Importing the Rankings for the end of the 2023-24 Season from the USAU website**

[usau rankings](https://play.usaultimate.org/teams/events/team_rankings/?RankSet=College-Men)

In [ ]:
url = "https://play.usaultimate.org/teams/events/team_rankings/?RankSet=College-Men"
team_rankings = pd.read_html("https://play.usaultimate.org/teams/events/team_rankings/?RankSet=College-Men", attrs = {'class': 'global_table'})

In [ ]:
team_rankings = pd.DataFrame(team_rankings[0])
team_rankings

,Rank,Team,Power Rating,Competition Level,Gender Division,Competition Division,College Region,College Conference,Wins,Losses
0,1,North Carolina,2124,College,Men,Division I,Atlantic Coast,Carolina DI,41,2
1,2,Massachusetts,2058,College,Men,Division I,New England,Greater New England DI,31,5
2,3,Cal Poly-SLO,1992,College,Men,Division I,Southwest,SoCal DI,39,7
3,4,Colorado,1969,College,Men,Division I,South Central,Rocky Mountain DI,29,6
4,5,Vermont,1940,College,Men,Division I,New England,Greater New England DI,26,11
5,6,Oregon,1923,College,Men,Division I,Northwest,Cascadia DI,36,11
6,7,Brown,1903,College,Men,Division I,New England,Greater New England DI,24,11
7,8,Pittsburgh,1898,College,Men,Division I,Ohio Valley,West Penn DI,27,8
8,9,California-Santa Cruz,1880,College,Men,Division I,Southwest,NorCal DI,33,10
9,10,Texas,1855,College,Men,Division I,South Central,South Texas DI,27,9


In [ ]:
team_rankings.set_index("Rank", inplace=True)

Saving the team rankings to a csv in our shared drive for later use:

In [ ]:
team_rankings = team_rankings.iloc[:20:]
team_rankings
team_rankings.to_csv("team_rankings")

##Selenium

**In order to scrape all of the articles in question, we will need to use a webdriver**

**These are sources I used to figure out how to use Selenuim:**

Resources:

*   https://nariyoo.com/python-how-to-run-selenium-in-google-colab/
*   https://stackoverflow.com/questions/51046454how-can-we-use-selenium-webdriver-in-colab-research-google-com

*   https://www.selenium.dev/documentation/webdriver/

*   https://www.selenium.dev/documentation/webdriver/getting_started/first_script/
*   https://www.selenium.dev/documentation/webdriver/elements/interactions/








In [ ]:
pip install selenium

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 27.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 448.3/448.3 kB 18.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.3/58.3 kB 7.9 MB/s eta 0:00:00


configuring a webdriver for selenium to use. This enables us to interact with a webpage we fetch

In [ ]:
%%shell
# Ubuntu no longer distributes chromium-browser outside of snap
#
# Proposed solution: https://askubuntu.com/questions/1204571/how-to-install-chromium-without-snap

# Add debian buster
cat > /etc/apt/sources.list.d/debian.list <<'EOF'
deb [arch=amd64 signed-by=/usr/share/keyrings/debian-buster.gpg] http://deb.debian.org/debian buster main
deb [arch=amd64 signed-by=/usr/share/keyrings/debian-buster-updates.gpg] http://deb.debian.org/debian buster-updates main
deb [arch=amd64 signed-by=/usr/share/keyrings/debian-security-buster.gpg] http://deb.debian.org/debian-security buster/updates main
EOF

# Add keys
apt-key adv --keyserver keyserver.ubuntu.com --recv-keys DCC9EFBF77E11517
apt-key adv --keyserver keyserver.ubuntu.com --recv-keys 648ACFD622F3D138
apt-key adv --keyserver keyserver.ubuntu.com --recv-keys 112695A0E562B32A

apt-key export 77E11517 | gpg --dearmour -o /usr/share/keyrings/debian-buster.gpg
apt-key export 22F3D138 | gpg --dearmour -o /usr/share/keyrings/debian-buster-updates.gpg
apt-key export E562B32A | gpg --dearmour -o /usr/share/keyrings/debian-security-buster.gpg

# Prefer debian repo for chromium* packages only
# Note the double-blank lines between entries
cat > /etc/apt/preferences.d/chromium.pref << 'EOF'
Package: *
Pin: release a=eoan
Pin-Priority: 500


Package: *
Pin: origin "deb.debian.org"
Pin-Priority: 300


Package: chromium*
Pin: origin "deb.debian.org"
Pin-Priority: 700
EOF

# Install chromium and chromium-driver
apt-get update
apt-get install chromium chromium-driver

# Install selenium
pip install selenium

Executing: /tmp/apt-key-gpghome.q2eFKkvs4o/gpg.1.sh --keyserver keyserver.ubuntu.com --recv-keys DCC9EFBF77E11517
gpg: key DCC9EFBF77E11517: public key "Debian Stable Release Key (10/buster) <debian-release@lists.debian.org>" imported
gpg: Total number processed: 1
gpg:               imported: 1
Executing: /tmp/apt-key-gpghome.GHwwWXMgJC/gpg.1.sh --keyserver keyserver.ubuntu.com --recv-keys 648ACFD622F3D138
gpg: key DC30D7C23CBBABEE: public key "Debian Archive Automatic Signing Key (10/buster) <ftpmaster@debian.org>" imported
gpg: Total number processed: 1
gpg:               imported: 1
Executing: /tmp/apt-key-gpghome.i9pOwBY5xC/gpg.1.sh --keyserver keyserver.ubuntu.com --recv-keys 112695A0E562B32A
gpg: key 4DFAB270CAA96DFA: public key "Debian Security Archive Automatic Signing Key (10/buster) <ftpmaster@debian.org>" imported
gpg: Total number processed: 1
gpg:               imported: 1
Get:1 http://deb.debian.org/debian buster InRelease [122 kB]
Get:2 http://deb.debian.org/debian bust




**Importing selenium and necessary functions.**

In [ ]:
from selenium import webdriver
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.by import By

In [ ]:
!apt-get update
!apt-get install chromium chromium-driver

Hit:1 http://deb.debian.org/debian buster InRelease
Hit:2 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:3 http://deb.debian.org/debian buster-updates InRelease
Hit:4 http://deb.debian.org/debian-security buster/updates InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:7 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:8 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:9 https://ppa.launchpadcontent.net/c2d4u.team/c2d4u4.0+/ubuntu jammy InRelease
Hit:10 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:11 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:12 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:13 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
Reading package lists... Done
Building dependency t

##Articles

**Scrape Ultiworld Articles about d1 college**

There are a pages on the ultiworld site containing links and graphics for each article related to the d1 college division. We want to navigate through each of these pages and find all of those links. Navigating through the pages and looking at the dates, IO was able to determine that the 19th page is as far is as relecant for our analysis.

[Here is the first page](https://ultiworld.com/division/usau-college-d-i-mens)


In [ ]:
article_links = []
first_page = "https://ultiworld.com/division/usau-college-d-i-mens"
response = requests.get(first_page)
soup = BeautifulSoup(response.content, "html.parser")

In [ ]:
articles = soup.find_all("h3", class_="snippet-excerpt__heading")
link = articles[0].find("a", href=True)['href']

In [ ]:
for article in articles:
  article_link = article.find("a", href=True)['href']
  article_links.append(article_link)
article_links

['https://ultiworld.com/2023/11/24/classic-city-classic-2023-tournament-recap/',
 'https://ultiworld.com/2023/11/17/classic-city-classic-2023-tournament-preview/',
 'https://ultiworld.com/2023/11/15/classic-city-classic-2023-streaming-schedule/',
 'https://ultiworld.com/2023/07/06/2023-mens-d-i-college-awards-snubs-superlatives/',
 'https://ultiworld.com/2023/06/27/ultiworlds-2023-d-i-college-awards-voting-breakdown/',
 'https://ultiworld.com/2023/06/23/2023-d-i-mens-coaches-of-the-year/',
 'https://ultiworld.com/2023/06/22/2023-d-i-mens-defensive-player-of-the-year/',
 'https://ultiworld.com/2023/06/21/2023-d-i-mens-offensive-player-of-the-year/',
 'https://ultiworld.com/2023/06/15/2023-d-i-mens-rookie-of-the-year/',
 'https://ultiworld.com/2023/06/14/2023-d-i-mens-breakout-player-of-the-year/']

In [ ]:
for idx in range(2, 19, 1):
  url = f"https://ultiworld.com/division/usau-college-d-i-mens/page/{idx}/"
  response = requests.get(url)
  soup = BeautifulSoup(response.content, "html.parser")
  articles = soup.find_all("h3", class_="snippet-excerpt__heading")
  for article in articles:
    article_link = article.find("a", href=True)['href']
    article_links.append(article_link)

In [ ]:
article_links

['https://ultiworld.com/2023/11/24/classic-city-classic-2023-tournament-recap/',
 'https://ultiworld.com/2023/11/17/classic-city-classic-2023-tournament-preview/',
 'https://ultiworld.com/2023/11/15/classic-city-classic-2023-streaming-schedule/',
 'https://ultiworld.com/2023/07/06/2023-mens-d-i-college-awards-snubs-superlatives/',
 'https://ultiworld.com/2023/06/27/ultiworlds-2023-d-i-college-awards-voting-breakdown/',
 'https://ultiworld.com/2023/06/23/2023-d-i-mens-coaches-of-the-year/',
 'https://ultiworld.com/2023/06/22/2023-d-i-mens-defensive-player-of-the-year/',
 'https://ultiworld.com/2023/06/21/2023-d-i-mens-offensive-player-of-the-year/',
 'https://ultiworld.com/2023/06/15/2023-d-i-mens-rookie-of-the-year/',
 'https://ultiworld.com/2023/06/14/2023-d-i-mens-breakout-player-of-the-year/',
 'https://ultiworld.com/2023/06/13/2023-d-i-mens-all-american-second-team/',
 'https://ultiworld.com/livewire/carnegie-mellons-wyatt-brooks-for-callahan-2023/',
 'https://ultiworld.com/2023/06

Now that we have a list of links to every article we want, we will scrape the text from the body of each article. First we define and configure an instance of a webdriver:

In [ ]:
from selenium import webdriver
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.by import By
options = webdriver.ChromeOptions()
options.add_argument("--verbose")
options.add_argument('--no-sandbox')
options.add_argument('--headless')
options.add_argument('--disable-gpu')
options.add_argument("--window-size=1920, 1200")
options.add_argument('--disable-dev-shm-usage')
options.add_argument("--dns-prefetch-disable")
driver = webdriver.Chrome(options=options)

Unfortunately, ulti world articles are held behind a pay wall for most of the season, and their site notoriously does not remember your login info. I bought a cheap access account for the project and used it to scrape the articles. Once we fetch and wait for a page to load, we check to see if the login prompt is preventing us from reading the article. If it is, we click on the login link, fill in our info, return to the article in question, and proceed to scrape the author, date, title, text. Repeat the process for a couple hundred links while taking a nap and it finishes running in just under 50 minutes!

In [ ]:
import time

texts = []
authors = []
dates = []
titles = []
count = 0

al = ['https://ultiworld.com/2023/01/10/bold-predictions-for-the-2023-college-season/']
def login(driver, link):
  driver.get(link)
  driver.find_element(By.LINK_TEXT, "Log in").click()
  driver.refresh()
  time.sleep(2)
  user_input = driver.find_element(By.ID, "user_login")
  user_input.send_keys("36triangles")
  pass_input = driver.find_element(By.ID, "user_pass")
  pass_input.send_keys("data301!")
  driver.find_element(By.ID, "wp-submit").click()

login(driver, article_links[0])
for link in article_links[1::]:
  try:
    login(driver, link)
  except:
    print("already loged in")
  driver.get(link)
  time.sleep(2)

  try:
    text = driver.find_element(By.XPATH, "/html/body/main/div[1]/div[2]").text
  except:
    text = 'None'

  try:
    author = driver.find_element(By.XPATH, "/html/body/main/div[1]/div[3]/ol/li/h5").text
  except:
    author = 'None'

  try:
    date = driver.find_element(By.XPATH, "/html/body/main/div[1]/div[2]/h6").text
  except:
    date = 'None'

  try:
    title = driver.find_element(By.XPATH, "/html/body/main/div[1]/div[1]/h1").text
  except:
    title = "None"

  texts.append(text)
  authors.append(author)
  dates.append(date)
  titles.append(title)



Finally, we get to put all the articles and their attributes into a dataframe, clean it, and save it for later use:

In [ ]:
df_articles = pd.DataFrame()
df_articles["author"] = authors
df_articles["text"] = texts
df_articles["date"] = dates
df_articles["title"] = titles
df_articles

,author,text,date,title
0,EDWARD STEPHENS,The college preseason pillar is back!\nNOVEMBE...,"NOVEMBER 17, 2023 BY EDWARD STEPHENS IN PREVIE...",Classic City Classic 2023: Tournament Preview
1,AIDAN SHAPIRO-LEIGHTON,The competitive college fall is here!\nNOVEMBE...,"NOVEMBER 15, 2023 BY AIDAN SHAPIRO-LEIGHTON IN...",Classic City Classic 2023: Streaming Schedule
2,ULTIWORLD,Our reporters celebrate some of their favorite...,"JULY 6, 2023 BY ULTIWORLD IN AWARDS WITH 0 COM...",2023 Men’s D-I College Awards: Snubs & Superla...
3,ULTIWORLD,See all of the vote-getters for every 2023 D-I...,"JUNE 27, 2023 BY ULTIWORLD IN AWARDS WITH 0 CO...",Ultiworld’s 2023 D-I College Awards Voting Bre...
4,JAKE THORNE,Celebrating the best off-field leaders of the ...,"JUNE 23, 2023 BY JAKE THORNE, ALEX RUBIN AND P...",2023 D-I Men’s Coaches of the Year
...,...,...,...,...
174,THEO WAN,The USA College Season is Officially here!\nJA...,"JANUARY 26, 2023 BY THEO WAN AND DANIE PROBY I...",Huckin’ Eh: Bellingham Recap & Santa Barbara P...
175,KEITH RAYNOR,Five teams from each division with all the goo...,"JANUARY 24, 2023 BY KEITH RAYNOR IN RANKINGS W...",10 Unranked Teams To Watch
176,EDWARD STEPHENS,We preview teams #1-5 in our Power Rankings he...,"JANUARY 20, 2023 BY EDWARD STEPHENS, PATRICK S...",2023 D-I College Preseason Power Rankings: #1-5
177,EDWARD STEPHENS,We preview teams #6-15 in our Power Rankings h...,"JANUARY 19, 2023 BY EDWARD STEPHENS, PATRICK S...",2023 D-I College Preseason Power Rankings: #6-15


In [ ]:
df_articles.date.value_counts()
def clean_date(x):
  str_list = x.split("BY")
  return str_list[0]
df_articles["date"] = df_articles["date"].map(clean_date)

In [ ]:
df_articles["date"]
df_articles.to_csv("articles", sep=',', index=False, encoding='utf-8')

In [ ]:
potato = pd.read_csv("articles")



##Data Description

**Data we have collected**

So far, we have scraped and cleaned the Cal Poly 2023 men's frisbee roster, a table containing information on every team in the men's d1 college division. Each table's variables are team ranking, power rating, region, name, and record for the 2023 season. We have also scraped a list of news article links from Ultiworld, a site that reports on college frisbee. Each link represents an article written by Ultiworld about the men's d1 college division in the 2023 season, and each individual article is our unit of measurement.

**Data we still have to collect**

Although we have Cal Poly's roster for the 2023 season, we still have to scrape the rosters of all the other teams. These rosters contain player's first and last names. We intend to get the rosters from the United States Ultimate website listed above. To access the page, the link requires an additional link click, so we are working on getting that. Also, we still have to loop through the ultiworld links and scrape the text from each article. From each article, we will be pulling the total word count, author, and record the number of times each college frisbee team is mentioned. To be mentioned, we are looking for the team name, players' name, or school name. This way, we can compare the number of times each team is mentioned, and see if Cal Poly's Slocore is mentiioned at a different proportion than other schools.